# ACTIVIDAD SESIÓN 4: PROCESAMIENTO DE DATOS CON SPARK SQL Y DATAFRAMES

El Ministerio de Educación publicó un informe con las **carreras con mayor cantidad de inscritos** en Chile en los últimos años.
Dispones de un dataset con: **carrera**, **universidad**, **inscritos** y **área**.  

**Objetivo**  
Procesar la información usando **Spark SQL** y **DataFrames** para responder preguntas clave y generar un informe procesado.

**Dataset**: `carreras.json`

## 1. Creación de la sesión Spark (1 punto)
- Crea una sesión de Spark con el nombre **"AnalisisCarreras"**.


In [12]:
from pyspark.sql import SparkSession

# Crear sesión de Spark

spark = SparkSession.builder.appName("AnalisisCarreras").getOrCreate()
spark
print("Spark inicializado")
print("Versión:", spark.version)


Spark inicializado
Versión: 4.0.0


## 2. Carga de datos en un DataFrame (1 punto)
- Carga los datos desde `carreras.json` en un **DataFrame de Spark**.  
- Muestra los **primeros registros**.


In [13]:
from pyspark.sql import functions as F, types as T

# Intentar leer desde carreras.json; si no existe, usar datos embebidos
try:
    df = spark.read.json("carreras.json", multiLine=True)
    origen = "archivo 'carreras.json'"
except Exception as e:
    data = [
        {"id": 1, "carrera": "Ingeniería Comercial", "universidad": "U. de Chile", "inscritos": 3200, "area": "Económicas y Administrativas"},
        {"id": 2, "carrera": "Derecho", "universidad": "PUC", "inscritos": 2900, "area": "Ciencias Sociales"},
        {"id": 3, "carrera": "Medicina", "universidad": "U. de Concepción", "inscritos": 2500, "area": "Salud"},
        {"id": 4, "carrera": "Psicología", "universidad": "U. de Santiago", "inscritos": 1800, "area": "Ciencias Sociales"},
        {"id": 5, "carrera": "Ingeniería Civil", "universidad": "UTFSM", "inscritos": 3100, "area": "Ingeniería y Tecnología"},
    ]
    schema = T.StructType([
        T.StructField("id", T.IntegerType()),
        T.StructField("carrera", T.StringType()),
        T.StructField("universidad", T.StringType()),
        T.StructField("inscritos", T.IntegerType()),
        T.StructField("area", T.StringType()),
    ])
    df = spark.createDataFrame(data, schema=schema)
    origen = "datos embebidos (fallback)"

print(f"Origen de datos: {origen}")
df.show(truncate=False)

25/09/05 22:20:33 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: carreras.json.
java.io.FileNotFoundException: File carreras.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfu

Origen de datos: datos embebidos (fallback)
+---+--------------------+----------------+---------+----------------------------+
|id |carrera             |universidad     |inscritos|area                        |
+---+--------------------+----------------+---------+----------------------------+
|1  |Ingeniería Comercial|U. de Chile     |3200     |Económicas y Administrativas|
|2  |Derecho             |PUC             |2900     |Ciencias Sociales           |
|3  |Medicina            |U. de Concepción|2500     |Salud                       |
|4  |Psicología          |U. de Santiago  |1800     |Ciencias Sociales           |
|5  |Ingeniería Civil    |UTFSM           |3100     |Ingeniería y Tecnología     |
+---+--------------------+----------------+---------+----------------------------+



## 3. Exploración del DataFrame (1 punto)
- **Imprime el esquema** del DataFrame para visualizar los tipos de datos de cada columna.


In [14]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- carrera: string (nullable = true)
 |-- universidad: string (nullable = true)
 |-- inscritos: integer (nullable = true)
 |-- area: string (nullable = true)



## 4. Consultas con Spark SQL (3 puntos)
Realiza las siguientes consultas usando **Spark SQL**:
- **(a)** Obtener todas las carreras con **más de 2500** inscritos.  
- **(b)** Contar cuántas carreras pertenecen a **cada área** de conocimiento.  
- **(c)** Mostrar las **universidades** que ofrecen **más de una carrera** en la lista.


In [15]:
# 4) Registrar vista temporal
df.createOrReplaceTempView("carreras")

# a) Carreras con más de 2500 inscritos
q_a = spark.sql("""
SELECT id, carrera, universidad, inscritos, area
FROM carreras
WHERE inscritos > 2500
ORDER BY inscritos DESC
""")
print("a) Carreras con > 2500 inscritos")
q_a.show(truncate=False)


a) Carreras con > 2500 inscritos
+---+--------------------+-----------+---------+----------------------------+
|id |carrera             |universidad|inscritos|area                        |
+---+--------------------+-----------+---------+----------------------------+
|1  |Ingeniería Comercial|U. de Chile|3200     |Económicas y Administrativas|
|5  |Ingeniería Civil    |UTFSM      |3100     |Ingeniería y Tecnología     |
|2  |Derecho             |PUC        |2900     |Ciencias Sociales           |
+---+--------------------+-----------+---------+----------------------------+



In [16]:
# b) Cantidad de carreras por área
q_b = spark.sql("""
SELECT area, COUNT(*) AS cantidad_carreras
FROM carreras
GROUP BY area
ORDER BY cantidad_carreras DESC, area
""")
print("b) Conteo por área")
q_b.show(truncate=False)

b) Conteo por área
+----------------------------+-----------------+
|area                        |cantidad_carreras|
+----------------------------+-----------------+
|Ciencias Sociales           |2                |
|Económicas y Administrativas|1                |
|Ingeniería y Tecnología     |1                |
|Salud                       |1                |
+----------------------------+-----------------+



In [17]:
# c) Universidades que ofrecen más de una carrera en la lista
q_c = spark.sql("""
SELECT universidad, COUNT(*) AS n_carreras
FROM carreras
GROUP BY universidad
HAVING COUNT(*) > 1
ORDER BY n_carreras DESC, universidad
""")
print("c) Universidades con más de una carrera")
q_c.show(truncate=False)

c) Universidades con más de una carrera
+-----------+----------+
|universidad|n_carreras|
+-----------+----------+
+-----------+----------+



## 5. UDF: Clasificación por demanda (2 puntos)
Crea una **UDF** que clasifique las carreras según `inscritos`:
- **"Alta demanda"** si `inscritos > 3000`.  
- **"Media demanda"** si `2000 ≤ inscritos ≤ 3000`.  
- **"Baja demanda"** si `inscritos < 2000`.

Aplica la UDF al DataFrame y agrega la columna **`demanda`**.  
Muestra el DataFrame actualizado.


In [18]:
from pyspark.sql.functions import udf

@udf("string")
def clasificar_demanda(inscritos):
    if inscritos is None:
        return None
    if inscritos > 3000:
        return "Alta demanda"
    elif 2000 <= inscritos <= 3000:
        return "Media demanda"
    else:
        return "Baja demanda"

df_demanda = df.withColumn("demanda", clasificar_demanda(F.col("inscritos")))
df_demanda.show(truncate=False)

+---+--------------------+----------------+---------+----------------------------+-------------+
|id |carrera             |universidad     |inscritos|area                        |demanda      |
+---+--------------------+----------------+---------+----------------------------+-------------+
|1  |Ingeniería Comercial|U. de Chile     |3200     |Económicas y Administrativas|Alta demanda |
|2  |Derecho             |PUC             |2900     |Ciencias Sociales           |Media demanda|
|3  |Medicina            |U. de Concepción|2500     |Salud                       |Media demanda|
|4  |Psicología          |U. de Santiago  |1800     |Ciencias Sociales           |Baja demanda |
|5  |Ingeniería Civil    |UTFSM           |3100     |Ingeniería y Tecnología     |Alta demanda |
+---+--------------------+----------------+---------+----------------------------+-------------+



## 6. Guardado de datos en Parquet (2 puntos)
- Guarda el DataFrame transformado en un archivo Parquet llamado **`carreras_procesadas.parquet`**.


In [21]:
# Guardar como Parquet (sobrescribir si existe)
output_path = "carreras_procesadas.parquet"
df_demanda.write.mode("overwrite").parquet(output_path)
print(f"Datos guardados en: {output_path}")

Datos guardados en: carreras_procesadas.parquet


25/09/06 02:43:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 641143 ms exceeds timeout 120000 ms
25/09/06 02:43:54 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/06 02:44:01 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

## INSTRUCCIONES ADICIONALES
- **Puntos totales = 10**.  
- Comprime el archivo en **.zip** o **.rar**.  
- Sube el archivo a la **plataforma**.
